In [ ]:
import logging
import sys
from gitsource import GithubRepositoryDataReader

# Set up logging to display INFO or DEBUG messages
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

print("Starting the read process...")
files = reader.read()
print(f"Finished! Fetched {len(files)} files.")


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

def llm(prompt, model="gpt-5.4-mini"):
    response = openai_client.responses.create(
        model=model,
        input=prompt
    )
    return response.output_text

# Build searchable documents from fetched lesson files
documents = []
for idx, raw_file in enumerate(files):
    if isinstance(raw_file, dict):
        path = raw_file.get("path") or raw_file.get("filename") or f"lesson-{idx+1}"
        content = raw_file.get("content") or raw_file.get("text")
    else:
        path = getattr(raw_file, "path", None) or getattr(raw_file, "filename", None) or f"lesson-{idx+1}"
        content = getattr(raw_file, "content", None) or getattr(raw_file, "text", None)

    if not content:
        continue

    documents.append({
        "question": f"Lesson file content for {path}",
        "section": path,
        "answer": content,
        "course": "llm-zoomcamp",
    })

lesson_count = len(documents)

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
)
index.fit(documents)

def search(question, course="llm-zoomcamp", num_results=5):
    return index.search(
        question,
        boost_dict={"question": 2.0, "section": 0.5},
        filter_dict={"course": course},
        num_results=num_results,
    )

INSTRUCTIONS = """
Your task is to answer questions about the llm-zoomcamp lessons based on the provided context.
Use the context to provide an accurate answer. If the answer is not contained in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append("Section: " + str(doc.get("section", "")))
        lines.append("Q: " + str(doc.get("question", "")))
        answer_snippet = str(doc.get("answer", ""))
        lines.append("A: " + answer_snippet[:500].replace("\n", " "))
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    context += f"\n\nTotal lesson files found: {lesson_count}"
    return USER_PROMPT_TEMPLATE.format(question=question, context=context).strip()

def rag(question, model="gpt-5.4-mini"):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    return llm(f"{INSTRUCTIONS}\n\n{prompt}", model=model)

In [ ]:
question = "How many lessons are there?"
answer = rag(question)
print("Prompt:", question)
print("Answer:", answer)

Q2 

In [ ]:
from minsearch import Index

documents = []
for idx, raw_file in enumerate(files):
    if isinstance(raw_file, dict):
        filename = raw_file.get("path") or raw_file.get("filename") or f"file-{idx+1}"
        content = raw_file.get("content") or raw_file.get("text")
    else:
        filename = getattr(raw_file, "path", None) or getattr(raw_file, "filename", None) or f"file-{idx+1}"
        content = getattr(raw_file, "content", None) or getattr(raw_file, "text", None)

    if not content:
        continue

    documents.append({
        "filename": filename,
        "content": content,
    })

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query, num_results=5)
search_results

Q3

In [ ]:
!wget -q https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [ ]:
from importlib.util import spec_from_file_location
from pathlib import Path

spec = spec_from_file_location("rag_helper_2", Path("rag_helper_2.py"))
rag_helper_2 = spec.loader.load_module("rag_helper_2")

assistant = rag_helper_2.RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=INSTRUCTIONS,
    model="gpt-5.4-mini",
)

result = assistant.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("Answer:\n", result.answer)
print("Usage:\n", result.usage)
print(
    "Input tokens:",
    getattr(result.usage, "input_tokens", None)
)

Q4

In [ ]:
from gitsource import chunk_documents
# run in your notebook before calling chunk_documents
chunks = list(chunk_documents(documents, size=2000, step=1000))
print("Total chunks:", len(chunks))

if isinstance(chunks[0], dict):
    key = "content" if "content" in chunks[0] else next(iter(chunks[0].keys()))
    print("\nFirst chunk preview:\n", chunks[0][key][:500])
else:
    print("\nFirst chunk preview:\n", chunks[0][:500])

Q5

In [ ]:
from minsearch import Index

docs_for_index = []
for i, chunk in enumerate(chunks):
    if isinstance(chunk, dict):
        key = "content" if "content" in chunk else next(iter(chunk.keys()))
        text = chunk[key]
        filename = chunk.get("filename", f"chunk-{i+1}")
    else:
        text = chunk
        filename = f"chunk-{i+1}"

    docs_for_index.append({
        "content": text,
        "filename": filename,
    })

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(docs_for_index)

assistant_chunks = rag_helper_2.RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    instructions=INSTRUCTIONS,
    model="gpt-5.4-mini",
)

question = "How does the agentic loop keep calling the model until it stops?"
result = assistant_chunks.rag(question)

print("Question:", question)
print("Answer:\n", result.answer)
print("Usage:\n", result.usage)
print(
    "Input tokens:",
    getattr(result.usage, "input_tokens", None) or getattr(result.usage, "prompt_tokens", None)
)

Q6

In [ ]:
import time

print("Waiting 60 seconds to reset the OpenAI TPM bucket...")
time.sleep(60) 
print("...")

result = runner.loop(prompt=question, callback=callback)

In [ ]:
# Define a typed search tool for the chunked index and run a ToyAIKit agent
# The function below must include a type hint and a docstring so ToyAIKit
# can auto-generate the tool schema.

SEARCH_CALL_COUNT = {"count": 0}

def search_chunked(query: str) -> list[dict]:
    """
    Search the chunked index for entries matching `query` and return a list
    of result dictionaries (each with at least `content` and `filename`).

    ToyAIKit will read this docstring and the type hint to build the
    function's JSON schema automatically. The agent should call this
    function when it needs to look up course material.
    """
    SEARCH_CALL_COUNT["count"] += 1
    return chunk_index.search(query, num_results=5)

# Register the tool with ToyAIKit and run the agent loop
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search_chunked)

instructions = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

question = "How does the agentic loop work, and how is it different from plain RAG?"
result = runner.loop(prompt=question, callback=callback)

print("Final result:\n", getattr(result, "output_text", result))
print("Search calls triggered:", SEARCH_CALL_COUNT["count"])